# Join datasets
- Author: Bryan Bravo
- Created: 2026-03-28
## Import Libraries

In [1]:
import os
import sys

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# Visualize data for joined dataframes.
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
# in_path = 's3a://ml-project-s3-bronze/input_folder/'
in_path = 'processed_datasets/'
out_path = 'processed_datasets/'


## Custom Functions

## Query


In [4]:
# Read and display the first few rows of each dataset to verify they are loaded correctly.
df_list = ['acled', 'cpi', 'fred', 'gpr', 'lsci', 'oil', 'wb']
for df_name in df_list:
    globals()[f"{df_name}_df"] = spark.read.csv(f"{in_path}/{df_name}.csv", header=True, inferSchema=True)
    print(f"DataFrame: {df_name}")
    globals()[f"{df_name}_df"].show(5)


DataFrame: acled
+---------+-----+----+------+
|  country|month|year|events|
+---------+-----+----+------+
|australia|    1|2021|     2|
|australia|    2|2021|     0|
|australia|    3|2021|     0|
|australia|    4|2021|     0|
|australia|    5|2021|     1|
+---------+-----+----+------+
only showing top 5 rows

DataFrame: cpi
+---------+-----+----+-----+
|  country|value|year|month|
+---------+-----+----+-----+
|australia| 96.4|2024|    4|
|australia|96.17|2024|    5|
|australia|96.52|2024|    6|
|australia|96.77|2024|    7|
|australia|96.49|2024|    8|
+---------+-----+----+-----+
only showing top 5 rows

DataFrame: fred
+--------+---------+-------------+-------+
|    date|  country|interest_rate|fx_rate|
+--------+---------+-------------+-------+
|20060403|australia|          5.5| 0.7177|
|20060404|australia|          5.5| 0.7203|
|20060405|australia|          5.5| 0.7263|
|20060406|australia|          5.5| 0.7315|
|20060407|australia|          5.5| 0.7279|
+--------+---------+-------

## Join Datasets and Cleaning
### FRED and Oil datasets
Joining and checking for duplicates

In [5]:
# FRED and oil
joined_df1 = fred_df.join(oil_df, on='date', how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df1.count(), len(joined_df1.columns))}")

## Check for duplicates
duplicate_count = joined_df1.groupBy(joined_df1.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df1: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 6)
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df1: 0


No duplicate rows were found.
checking for missing values

In [6]:
## Check for missing values
missing_values = {col: joined_df1.filter(joined_df1[col].isNull()).count() for col in joined_df1.columns}
print("Missing values in joined_df1:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df1:
--- --- --- --- --- --- --- --- ---
date: 0
country: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 1278
wti_dollars_per_barrel: 1278


In [ ]:

# Visualize missing data in the joined dataframe.
# Note: This will convert the Spark DataFrame to a Pandas DataFrame, which may not be efficient for large datasets.
msno.matrix(joined_df1.toPandas(), labels=True)
plt.show()


#### Missing data treatment
A close inspection of the dates in relation to the missing oil-price data reveals that they fall on **market holidays or non-trading days**. Oil benchmarks do not publish prices on these days. This is expected behavior.
- Using F-Fill on the missing data is applicable in this situation based on the assumption that the _true price_ is the last traded price for this analysis.

In [7]:
(joined_df1.filter(F.col('brent_dollars_per_barrel').isNull()
                   | F.col('wti_dollars_per_barrel').isNull())
.orderBy('country', 'date').show())

+--------+---------+----------------+-------+------------------------+----------------------+
|    date|  country|   interest_rate|fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|
+--------+---------+----------------+-------+------------------------+----------------------+
|20060414|australia|             5.5| 0.7289|                    NULL|                  NULL|
|20060417|australia|             5.5| 0.7375|                    NULL|                  NULL|
|20060703|australia|            5.75| 0.7436|                    NULL|                  NULL|
|20061029|australia|             6.0| 0.7692|                    NULL|                  NULL|
|20061124|australia|6.19318181818182| 0.7782|                    NULL|                  NULL|
|20061226|australia|            6.25| 0.7829|                    NULL|                  NULL|
|20070406|australia|            6.25| 0.8177|                    NULL|                  NULL|
|20070409|australia|            6.25| 0.8167|               

In [8]:
# F-Fill missing values for oil prices in joined_df1.
w1 = W.partitionBy('country').orderBy('date')
joined_df1 = (
    joined_df1
    .withColumns({
        col: F.last(F.col(col), ignorenulls=True).over(w1) for col in ['brent_dollars_per_barrel', 'wti_dollars_per_barrel']
    })
)

## Check for missing values
missing_values = {col: joined_df1.filter(joined_df1[col].isNull()).count() for col in joined_df1.columns}
print("Missing values in joined_df1:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")

Missing values in joined_df1:
--- --- --- --- --- --- --- --- ---
date: 0
country: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0


#### Check for outliers
Due to how dynamic the data is for a 10 year period, the outliers will be checked per country on a yearly basis


In [32]:
# Define a function for outlier detection using the IQR method.
def outlier_detection_iqr(df, column):
    q1 = df.approxQuantile(column, [0.25], 0.01)[0]
    q3 = df.approxQuantile(column, [0.75], 0.01)[0]
    iqr = q3 - q1
    lower_threshold = q1 - 1.5 * iqr
    upper_threshold = q3 + 1.5 * iqr

    outlier_rows = df.filter((F.col(column) < lower_threshold) | (F.col(column) > upper_threshold))
    return outlier_rows



In [33]:
# Check for IQR outliers in per country and year in numeric columns
country_list = joined_df1.select('country').distinct().rdd.flatMap(lambda x: x).collect()
numeric_cols = ['interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel']
year_range = joined_df1.withColumn('date_sp', F.to_date('date', 'yyyyMMdd')).select(F.min(F.year('date_sp')), F.max(F.year('date_sp'))).collect()[0]
year_range = [year_range[0], year_range[1]]

for country in country_list:
    for year in range(year_range[0], year_range[1] + 1):
        subset_df = joined_df1.filter((F.col('country') == country) & (F.year(F.to_date('date', 'yyyyMMdd')) == year))
        print(f"Outliers for {country} in {year}:")
        for col in numeric_cols:
            outliers = outlier_detection_iqr(subset_df, col)
            outlier_count = outliers.count()
            print(f"  {col}: {outlier_count} outliers")

Outliers for brazil in 2006:
  interest_rate: 0 outliers
  fx_rate: 20 outliers
  brent_dollars_per_barrel: 0 outliers
  wti_dollars_per_barrel: 0 outliers
Outliers for brazil in 2007:
  interest_rate: 0 outliers
  fx_rate: 0 outliers
  brent_dollars_per_barrel: 0 outliers
  wti_dollars_per_barrel: 0 outliers
Outliers for brazil in 2008:
  interest_rate: 0 outliers
  fx_rate: 0 outliers
  brent_dollars_per_barrel: 16 outliers
  wti_dollars_per_barrel: 11 outliers
Outliers for brazil in 2009:
  interest_rate: 0 outliers
  fx_rate: 0 outliers
  brent_dollars_per_barrel: 0 outliers
  wti_dollars_per_barrel: 0 outliers
Outliers for brazil in 2010:
  interest_rate: 0 outliers
  fx_rate: 0 outliers
  brent_dollars_per_barrel: 0 outliers
  wti_dollars_per_barrel: 0 outliers
Outliers for brazil in 2011:
  interest_rate: 0 outliers
  fx_rate: 0 outliers
  brent_dollars_per_barrel: 7 outliers
  wti_dollars_per_barrel: 0 outliers
Outliers for brazil in 2012:
  interest_rate: 0 outliers
  fx_rate:

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "C:\spark\spark-3.5.1-bin-hadoop3\python\lib\py4j-0.10.9.7-src.zip\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\spark\spark-3.5.1-bin-hadoop3\python\lib\py4j-0.10.9.7-src.zip\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bravo\anaconda3\envs\ml_project_local\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [9]:
# print(f"Joined DataFrame (row, col) count: {(joined_df6.count(), len(joined_df6.columns))}")
# # print(joined_df6.toPandas().info(memory_usage='deep'))  # memory usage 447.3 MB 
# joined_df6.show(5)